# BÀI THỰC HÀNH 5: KIẾN TRÚC TRANSFORMER ENCODER

<b>Hướng dẫn nộp bài:</b> Các bạn commit và push code lên github, sử dụng file txt đặt tên theo cú pháp <MSSV>.txt chứa đường link dẫn đến github của bài thực hành và nộp file txt này tên courses.

### Bài 1: Xây dựng mô hình Transformer Encoder gồm 3 lớp theo mô tả trong nghiên cứu [Attention is all you need](https://proceedings.neurips.cc/paper_files/paper/2017/file/3f5ee243547dee91fbd053c1c4a845aa-Paper.pdf). Huấn luyện mô hình này cho bài toán phân loại domain câu bình luận trên bộ dữ liệu [UIT-ViOCD](https://drive.google.com/drive/folders/1Lu9axyLkw7dMx80uLRgvCnZsmNzhJWAa?usp=sharing).

### Bài 2: Xây dựng mô hình Transformer Encoder gồm 3 lớp theo mô tả trong nghiên cứu [Attention is all you need](https://proceedings.neurips.cc/paper_files/paper/2017/file/3f5ee243547dee91fbd053c1c4a845aa-Paper.pdf). Huấn luyện mô hình này cho bài toán gán nhãn chuỗi trên bộ dữ liệu [PhoNERT](https://github.com/VinAIResearch/PhoNER_COVID19).

---

## HƯỚNG DẪN THỰC HIỆN

### Cấu trúc code đã chuẩn bị:

```
Lab5/
├── src/
│   ├── models/
│   │   └── transformer_encoder.py      # ✓ Đã implement
│   ├── data/
│   │   ├── uit_viocd_dataset.py       # ✓ Đã implement
│   │   └── phonert_dataset.py         # ✓ Đã implement
│   ├── training/
│   │   └── trainer.py                 # ✓ Đã implement
│   └── utils/
│       └── utils.py                   # ✓ Đã implement
├── run_kaggle.ipynb                   # ✓ Main notebook
├── test_components.py                 # ✓ Test script
├── requirements.txt                   # ✓ Dependencies
└── README.md                          # ✓ Hướng dẫn chi tiết
```

### Các bước thực hiện:

#### 1. Test code local (optional)
```bash
cd Lab5
pip install -r requirements.txt
python test_components.py
```

#### 2. Push code lên GitHub
```bash
git add .
git commit -m "Lab5: Transformer Encoder implementation"
git push origin main
```

#### 3. Chạy trên Kaggle
- Mở **run_kaggle.ipynb** trên Kaggle
- Sửa dòng clone GitHub với repo của bạn
- Bật GPU (Settings → Accelerator → GPU)
- Run All Cells

#### 4. Nộp bài
- Tạo file **MSSV.txt** chứa link GitHub
- Nộp file txt lên courses

---

## CHI TIẾT IMPLEMENTATION

### Model Architecture (Đã implement trong transformer_encoder.py)

#### 1. Positional Encoding
- Sử dụng sin/cos functions theo công thức trong paper
- PE(pos, 2i) = sin(pos / 10000^(2i/d_model))
- PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))

#### 2. Multi-Head Attention
- 8 attention heads
- Scaled dot-product attention: Attention(Q,K,V) = softmax(QK^T / sqrt(d_k))V
- Concat heads và linear projection

#### 3. Transformer Encoder Layer (x3)
Mỗi layer gồm:
- Multi-Head Self-Attention
- Add & Norm (Residual + Layer Normalization)
- Position-wise Feed-Forward Network
- Add & Norm

#### 4. Classification/Token Classification Head
- **Bài 1**: Mean pooling → Linear layer → Softmax
- **Bài 2**: Linear layer cho mỗi token → Softmax

### Hyperparameters

```python
D_MODEL = 256          # Embedding dimension
NUM_HEADS = 8          # Attention heads
NUM_LAYERS = 3         # Encoder layers
D_FF = 1024           # Feed-forward dimension
DROPOUT = 0.1         # Dropout rate
MAX_LEN = 128         # Max sequence length
BATCH_SIZE = 32       # Batch size
LEARNING_RATE = 1e-4  # Learning rate
```

### Training Strategy

- **Optimizer**: AdamW with weight decay 0.01
- **Scheduler**: ReduceLROnPlateau (giảm LR khi dev metric không cải thiện)
- **Early Stopping**: Dừng sau 5 epochs không cải thiện
- **Gradient Clipping**: max_norm = 1.0

### Metrics

**Bài 1 (Classification)**:
- Accuracy (main metric)
- Precision, Recall, F1-Score (macro average)

**Bài 2 (NER)**:
- F1-Score (main metric) 
- Precision, Recall (macro average)
- Chỉ tính trên non-padding tokens

---

## DEMO CODE - Test nhanh local

### Test Model Architecture

In [ ]:
# Test model architecture
import sys
sys.path.append('src')

import torch
from models.transformer_encoder import TransformerForSequenceClassification
from utils.utils import print_model_info

# Create a small model for testing
model = TransformerForSequenceClassification(
    vocab_size=1000,
    num_classes=5,
    d_model=128,
    num_heads=4,
    num_layers=2,
    d_ff=512,
    max_len=64,
    dropout=0.1
)

print_model_info(model)

# Test forward pass
batch_size = 2
seq_len = 32
input_ids = torch.randint(0, 1000, (batch_size, seq_len))
attention_mask = torch.ones(batch_size, seq_len)

logits = model(input_ids, attention_mask)
print(f"\nOutput shape: {logits.shape}")
print(f"Expected: [{batch_size}, {model.classifier.out_features}]")
print("✓ Model works correctly!")

### Test Data Loading (nếu có data)

In [ ]:
# Test data loading (uncomment nếu có data)
"""
from data.uit_viocd_dataset import create_uit_viocd_dataloaders

# Paths
train_path = 'src/data/UIT_ViOCD/train_preprocessed.json'
dev_path = 'src/data/UIT_ViOCD/dev_preprocessed.json'
test_path = 'src/data/UIT_ViOCD/test_preprocessed.json'

# Create dataloaders
train_loader, dev_loader, test_loader, vocab, num_classes = create_uit_viocd_dataloaders(
    train_path, dev_path, test_path,
    batch_size=4,
    max_len=64
)

print(f"Vocab size: {len(vocab)}")
print(f"Number of classes: {num_classes}")
print(f"Train batches: {len(train_loader)}")

# Test một batch
for batch in train_loader:
    print(f"\nBatch shapes:")
    print(f"  Input IDs: {batch['input_ids'].shape}")
    print(f"  Attention mask: {batch['attention_mask'].shape}")
    print(f"  Labels: {batch['labels'].shape}")
    break
"""
print("Data loading code is ready!")

---

## LƯU Ý QUAN TRỌNG

### 1. Để chạy trên Kaggle:
- Sử dụng notebook **run_kaggle.ipynb**
- File đó đã cấu hình đầy đủ để clone từ GitHub và chạy
- Nhớ sửa link GitHub repo của bạn

### 2. Cấu trúc dữ liệu:
- **UIT-ViOCD**: Dùng file `*_preprocessed.json` (đã tiền xử lý)
- **PhoNERT**: Dùng file `.json` (không phải `.conll`)

### 3. Training time ước tính:
- **Bài 1**: ~15-20 phút trên GPU (Kaggle)
- **Bài 2**: ~30-40 phút trên GPU (dataset lớn hơn)

### 4. Giảm thời gian training (nếu cần):
```python
# Giảm hyperparameters
BATCH_SIZE = 64       # Tăng từ 32
D_MODEL = 128         # Giảm từ 256
NUM_LAYERS = 2        # Giảm từ 3
NUM_EPOCHS = 10       # Giảm từ 20
```

### 5. Expected results (tham khảo):
- **Bài 1**: Accuracy ~70-80%
- **Bài 2**: F1-Score ~80-85%

### 6. Files quan trọng cần nộp:
- Code trên GitHub (push tất cả files trong Lab5/)
- File MSSV.txt chứa link GitHub
- Không cần nộp checkpoints (file .pt quá lớn)